In [0]:
from pyspark.sql.functions import col, avg, lag, round
from pyspark.sql.window import Window

# 1. Read from Silver
silver_df = spark.table("main.market_project.silver_bank_prices")

# 2. Define Window for calculations
window_spec = Window.partitionBy("bank_name").orderBy("event_timestamp")

# 3. Business Logic / Aggregations
gold_df = silver_df.withColumn("prev_close_price", lag("close_price").over(window_spec)) \
    .withColumn("daily_return_pct", 
        round(((col("close_price") - col("prev_close_price")) / col("prev_close_price")) * 100, 2)) \
    .withColumn("moving_avg_7d", 
        round(avg("close_price").over(window_spec.rowsBetween(-6, 0)), 2)) \
    .fillna({"daily_return_pct": 0, "moving_avg_7d": 0}) \
    .select(
        "bank_name",
        "event_timestamp",
        "open_price",
        "close_price",
        "daily_return_pct",
        "moving_avg_7d",
        "volume"
    )

# 4. Save to Gold table
target_table = "main.market_project.gold_bank_metrics"
gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

print(f"Success: Gold table '{target_table}' created.")
display(gold_df.limit(500))